In [ ]:
#| default_exp store

In [ ]:
#| export
from __future__ import annotations

import json
import sqlite3
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Mapping

In [ ]:
#| export
@dataclass(frozen=True)
class RunManifest:
    run_id: str
    dataset_id: str
    ruleset_id: str
    calibration_id: str | None = None
    created_at: str = ""

    def __post_init__(self):
        if not self.run_id or not self.dataset_id or not self.ruleset_id: raise ValueError("run_id, dataset_id and ruleset_id are required")

In [ ]:
#| export
class RunStore:
    """SQLite persistence for immutable design-run records."""
    def __init__(self, path: str | Path):
        self.connection = sqlite3.connect(str(path))
        self.connection.execute("CREATE TABLE IF NOT EXISTS runs (run_id TEXT PRIMARY KEY, manifest_json TEXT NOT NULL)")
        self.connection.execute("CREATE TABLE IF NOT EXISTS records (run_id TEXT NOT NULL, candidate_id TEXT NOT NULL, payload_json TEXT NOT NULL, PRIMARY KEY(run_id, candidate_id))")

    def create_run(self, manifest: RunManifest) -> None:
        value = asdict(manifest); value["created_at"] = value["created_at"] or datetime.now(timezone.utc).isoformat()
        with self.connection: self.connection.execute("INSERT INTO runs VALUES (?, ?)", (manifest.run_id, json.dumps(value, sort_keys=True)))

    def add_records(self, run_id: str, records: Iterable[Mapping[str, object]]) -> None:
        rows = [(run_id, str(record["candidate_id"]), json.dumps(dict(record), sort_keys=True)) for record in records]
        with self.connection: self.connection.executemany("INSERT INTO records VALUES (?, ?, ?)", rows)

    def get_records(self, run_id: str) -> list[dict]:
        return [json.loads(row[0]) for row in self.connection.execute("SELECT payload_json FROM records WHERE run_id=? ORDER BY candidate_id", (run_id,))]